# INST447 — Week 3: Pandas: Aggregate, Group, Join

Lecture notes and study notebook · Fall 2026 · Wei Ai

Source: `week03_pandas_aggregate_group_join.html`. These are study aids, not an official assignment or submission. Explanations paraphrase the lecture; added exercises are labeled supplemental.

Open in Jupyter, VS Code, or Colab. Use **Restart Kernel and Run All** to check execution order. Weeks 2–3 need pandas: if missing, run `%pip install pandas` in a new cell, then restart. All datasets are embedded; no API keys, paid tools, or downloads are needed. Saved output was produced by executing the Python cells in order.

Study routine: read the explanation, predict the output, run the example, and try the exercise before viewing its solution.

## Setup and learning goals

Summarize a table; explain missing-value denominators; count categories; group and aggregate; join on keys; prevent duplicated lookup matches; and distinguish airline-weighted from flight-weighted averages. Reshaping appears in the tentative syllabus, but melt/pivot are not covered in this uploaded lecture and are not added as lecture content.

In [1]:
import pandas as pd

def display(*objects):
    for obj in objects:
        print(obj)
        print()

flights_data = [
    ("2024-01-15", "UA1247", "BWI", "ORD", "651", "B737", "12A", 289.50, 15),
    ("2024-01-22", "DL456", "ORD", "LAX", "1745", "A321", "8F", 425.00, None),
    ("2024-02-08", "WN2891", "LAX", "PHX", "370", "B737", "", 149.99, 0),
    ("2024-02-10", "WN1055", "PHX", "DEN", "602", "B737", "15C", None, 45),
    ("2024-03-05", "AA892", "DEN", "DFW", "663", "B737", "21B", 198.75, None),
    ("2024-03-12", "UA634", "DFW", "IAD", "1216", "B777", "9A", 345.25, 12),
    ("2024-04-20", "B61840", "IAD", "BOS", "429", "", "11D", 179.50, 0),
    ("2024-05-15", "DL1123", "BOS", "ATL", "946", "A220", "4A", 267.00, 25),
    ("2024-05-18", "DL2967", "ATL", "MIA", "594", "B737", "", None, 8),
    ("2024-06-02", "AA1456", "MIA", "LGA", "1095", "A321", "18F", 312.80, None),
]

columns = [
    "flight_date",
    "flight_number",
    "origin",
    "destination",
    "distance",
    "aircraft",
    "seat",
    "price",
    "delay_min",
]

flights = pd.DataFrame(flights_data, columns=columns)
display(flights)
flights['distance'] = flights['distance'].astype(int)

  flight_date flight_number origin destination  ... aircraft seat   price  delay_min
0  2024-01-15        UA1247    BWI         ORD  ...     B737  12A  289.50       15.0
1  2024-01-22         DL456    ORD         LAX  ...     A321   8F  425.00        NaN
2  2024-02-08        WN2891    LAX         PHX  ...     B737       149.99        0.0
3  2024-02-10        WN1055    PHX         DEN  ...     B737  15C     NaN       45.0
4  2024-03-05         AA892    DEN         DFW  ...     B737  21B  198.75        NaN
5  2024-03-12         UA634    DFW         IAD  ...     B777   9A  345.25       12.0
6  2024-04-20        B61840    IAD         BOS  ...           11D  179.50        0.0
7  2024-05-15        DL1123    BOS         ATL  ...     A220   4A  267.00       25.0
8  2024-05-18        DL2967    ATL         MIA  ...     B737          NaN        8.0
9  2024-06-02        AA1456    MIA         LGA  ...     A321  18F  312.80        NaN

[10 rows x 9 columns]



## Descriptive statistics

`describe()` gives numerical summaries by default. `include="all"` includes text summaries such as unique counts and most frequent values. Statistics that do not apply to a field appear missing. Use `sum`, `mean`, `max`, and `min` for specific questions.

In [2]:
display(flights.describe())
display(flights[['aircraft','seat','distance','price','delay_min']].describe(include='all'))
print('Flights:', len(flights))
print('Total miles:', flights['distance'].sum())
print('Mean miles:', flights['distance'].mean())
print('Longest / shortest:', flights['distance'].max(), flights['distance'].min())
assert flights['distance'].sum() == 8311

          distance       price  delay_min
count    10.000000    8.000000   7.000000
mean    831.100000  270.973750  15.000000
std     422.939568   92.249878  15.853496
min     370.000000  149.990000   0.000000
25%     596.000000  193.937500   4.000000
50%     657.000000  278.250000  12.000000
75%    1057.750000  320.912500  20.000000
max    1745.000000  425.000000  45.000000

       aircraft seat     distance       price  delay_min
count        10   10    10.000000    8.000000   7.000000
unique        5    9          NaN         NaN        NaN
top        B737               NaN         NaN        NaN
freq          5    2          NaN         NaN        NaN
mean        NaN  NaN   831.100000  270.973750  15.000000
std         NaN  NaN   422.939568   92.249878  15.853496
min         NaN  NaN   370.000000  149.990000   0.000000
25%         NaN  NaN   596.000000  193.937500   4.000000
50%         NaN  NaN   657.000000  278.250000  12.000000
75%         NaN  NaN  1057.750000  320.912500  20.0

## Missingness, counts, and axes

`len(df)` counts rows; `Series.count()` counts nonmissing values. `isna()` creates a Boolean table and summing it counts missing values. With `axis=0` (default), sum down each column; with `axis=1`, sum across each row. Empty strings are not counted as missing.

In [3]:
print('Prices recorded:', flights['price'].count(), 'out of', len(flights))
display(flights.isna(), flights.isna().sum(), flights.isna().sum(axis=1))
assert flights.isna().sum(axis=1).equals(flights.isna().apply(sum, axis=1))

Prices recorded: 8 out of 10
   flight_date  flight_number  origin  ...   seat  price  delay_min
0        False          False   False  ...  False  False      False
1        False          False   False  ...  False  False       True
2        False          False   False  ...  False  False      False
3        False          False   False  ...  False   True      False
4        False          False   False  ...  False  False       True
5        False          False   False  ...  False  False      False
6        False          False   False  ...  False  False      False
7        False          False   False  ...  False  False      False
8        False          False   False  ...  False   True      False
9        False          False   False  ...  False  False       True

[10 rows x 9 columns]

flight_date      0
flight_number    0
origin           0
destination      0
distance         0
aircraft         0
seat             0
price            2
delay_min        3
dtype: int64

0    0
1    1


## The denominator changes the meaning

The recorded delays sum to 105 minutes. There are seven known delays, so their mean is 15 minutes. Filling three unknown delays with zero divides the same total by ten, giving 10.5. `fillna(0)` does not establish that unknown flights were on time; it changes the assumption.

In [4]:
print('Known-delay mean:', flights['delay_min'].mean())
print('Zero-filled mean:', flights['delay_min'].fillna(0).mean())
print('Known count:', flights['delay_min'].count())
assert flights['delay_min'].mean() == 15
assert flights['delay_min'].fillna(0).mean() == 10.5

Known-delay mean: 15.0
Zero-filled mean: 10.5
Known count: 7


## Seat inference and its limits

A pattern such as `[0-9]+[AF]` identifies candidate window seats under an assumed ABC/DEF cabin layout. A different cabin can have different window letters. Seat labels alone do not prove seat type, and observed assignments do not prove passenger preference.

The lecture contrasts a local rule, structured seat-map API, LLM with a provided map, LLM retrieving a map, and agent-planned lookup. Each boundary adds dependencies and uncertainty. The example seat-map URL is illustrative, not a working service. No network calls are made here.

In [5]:
seats = pd.Series(['12A','8F','15C','21B'])
display(seats.str.fullmatch(r'[0-9]+[AF]'))

0     True
1     True
2    False
3    False
dtype: bool



## Unique values and category frequencies

Extract airline codes using `.str[:2]`. `unique()` gives distinct recorded values; `nunique()` counts distinct nonmissing values by default. `value_counts()` counts categories and ordinarily excludes missing values. Here B6 is one code even though it contains a digit.

In [6]:
flights['airline_code'] = flights['flight_number'].str[:2]
display(flights[['flight_number','airline_code']], flights['airline_code'].unique(), flights['airline_code'].value_counts())
assert flights['airline_code'].nunique() == 5

  flight_number airline_code
0        UA1247           UA
1         DL456           DL
2        WN2891           WN
3        WN1055           WN
4         AA892           AA
5         UA634           UA
6        B61840           B6
7        DL1123           DL
8        DL2967           DL
9        AA1456           AA

['UA' 'DL' 'WN' 'AA' 'B6']

airline_code
DL    3
UA    2
WN    2
AA    2
B6    1
Name: count, dtype: int64



## Grouping changes what one output row means

`groupby()` creates a grouping object; an aggregation computes the result. Group by airline to obtain one summary per airline. `.size()` counts all group rows, while a selected column’s `.count()` counts its nonmissing values. An airline with entirely unknown delays has a missing mean, not a zero mean.

In [7]:
groups = flights.groupby('airline_code')
display(groups.size(), groups['distance'].sum(), groups['distance'].max().reset_index())
display(groups['price'].count(), groups['delay_min'].mean())

airline_code
AA    2
B6    1
DL    3
UA    2
WN    2
dtype: int64

airline_code
AA    1758
B6     429
DL    3285
UA    1867
WN     972
Name: distance, dtype: int64

  airline_code  distance
0           AA      1095
1           B6       429
2           DL      1745
3           UA      1216
4           WN       602

airline_code
AA    2
B6    1
DL    2
UA    2
WN    1
Name: price, dtype: int64

airline_code
AA     NaN
B6     0.0
DL    16.5
UA    13.5
WN    22.5
Name: delay_min, dtype: float64



## Several aggregates and hierarchical columns

A dictionary of aggregate lists creates a MultiIndex for the columns. Resetting the row index does not flatten these column levels. Flatten the columns before resetting, or use named aggregation.

In [8]:
airline_stats = flights.groupby('airline_code').agg({'distance':['count','sum','mean','max'], 'price':['mean','min','max'], 'delay_min':'mean'})
display(airline_stats, airline_stats.columns)
print('After reset_index:', airline_stats.reset_index().columns)
flat = airline_stats.copy()
flat.columns = ['_'.join(col).strip() for col in flat.columns]
display(flat.reset_index())

             distance                        price                 delay_min
                count   sum    mean   max     mean     min     max      mean
airline_code                                                                
AA                  2  1758   879.0  1095  255.775  198.75  312.80       NaN
B6                  1   429   429.0   429  179.500  179.50  179.50       0.0
DL                  3  3285  1095.0  1745  346.000  267.00  425.00      16.5
UA                  2  1867   933.5  1216  317.375  289.50  345.25      13.5
WN                  2   972   486.0   602  149.990  149.99  149.99      22.5

MultiIndex([( 'distance', 'count'),
            ( 'distance',   'sum'),
            ( 'distance',  'mean'),
            ( 'distance',   'max'),
            (    'price',  'mean'),
            (    'price',   'min'),
            (    'price',   'max'),
            ('delay_min',  'mean')],
           )

After reset_index: MultiIndex([('airline_code',      ''),
            (    'dist

## Named aggregation and multiple grouping keys

Named aggregation expresses each output name as `(input_column, operation)`. It avoids positional renaming and makes the calculation easier to review. Grouping by two fields changes the unit of the summary to each distinct pair.

In [9]:
airline_stats_named = flights.groupby('airline_code').agg(
    flight_count=('flight_number','size'), total_distance=('distance','sum'),
    avg_distance=('distance','mean'), max_distance=('distance','max'),
    avg_price=('price','mean'), min_price=('price','min'),
    max_price=('price','max'), avg_delay=('delay_min','mean')).reset_index()
display(airline_stats_named)
display(flights.groupby(['airline_code','origin']).size().reset_index(name='flight_count'))
display(flights.groupby(['airline_code','aircraft']).size().reset_index(name='flight_count'))

  airline_code  flight_count  total_distance  ...  min_price  max_price  avg_delay
0           AA             2            1758  ...     198.75     312.80        NaN
1           B6             1             429  ...     179.50     179.50        0.0
2           DL             3            3285  ...     267.00     425.00       16.5
3           UA             2            1867  ...     289.50     345.25       13.5
4           WN             2             972  ...     149.99     149.99       22.5

[5 rows x 9 columns]

  airline_code origin  flight_count
0           AA    DEN             1
1           AA    MIA             1
2           B6    IAD             1
3           DL    ATL             1
4           DL    BOS             1
5           DL    ORD             1
6           UA    BWI             1
7           UA    DFW             1
8           WN    LAX             1
9           WN    PHX             1

  airline_code aircraft  flight_count
0           AA     A321             1
1     

## Join keys and lookup data

A join matches key values, not row positions. The flight table has repeated airline codes; the airline lookup should have one row per code. The lecture deliberately omits B6 from the lookup and includes AS, which has no flights.

In [10]:
airlines = pd.DataFrame([('UA','United Airlines'),('DL','Delta Air Lines'),('WN','Southwest Airlines'),('AA','American Airlines'),('AS','Alaska Airlines')], columns=['code','airline_name'])
display(airlines)

  code        airline_name
0   UA     United Airlines
1   DL     Delta Air Lines
2   WN  Southwest Airlines
3   AA   American Airlines
4   AS     Alaska Airlines



## Four join types

| Join | Retained keys | Rows in this example |
|---|---|---|
| inner | Matched on both sides | 9 |
| left | Every flight key | 10 |
| right | Every lookup key | 10 |
| outer | Keys from either side | 11 |

B6 is left-only and AS is right-only. Missing fields appear for unmatched records. These row counts depend on the particular keys and multiplicities; a left join does not always preserve the original row count.

In [11]:
joins = {}
for how in ['inner','left','right','outer']:
    joins[how] = flights.merge(airlines,left_on='airline_code',right_on='code',how=how)
    print(how, 'rows:', len(joins[how]))
    display(joins[how])
assert [len(joins[h]) for h in ['inner','left','right','outer']] == [9,10,10,11]

inner rows: 9
  flight_date flight_number origin  ... airline_code  code        airline_name
0  2024-01-15        UA1247    BWI  ...           UA    UA     United Airlines
1  2024-01-22         DL456    ORD  ...           DL    DL     Delta Air Lines
2  2024-02-08        WN2891    LAX  ...           WN    WN  Southwest Airlines
3  2024-02-10        WN1055    PHX  ...           WN    WN  Southwest Airlines
4  2024-03-05         AA892    DEN  ...           AA    AA   American Airlines
5  2024-03-12         UA634    DFW  ...           UA    UA     United Airlines
6  2024-05-15        DL1123    BOS  ...           DL    DL     Delta Air Lines
7  2024-05-18        DL2967    ATL  ...           DL    DL     Delta Air Lines
8  2024-06-02        AA1456    MIA  ...           AA    AA   American Airlines

[9 rows x 12 columns]

left rows: 10
  flight_date flight_number origin  ... airline_code  code        airline_name
0  2024-01-15        UA1247    BWI  ...           UA    UA     United Airlines


## Duplicate matches and validation

If each of two UA flights matches two lookup entries, there are four matched UA rows. The left join grows from ten to twelve rows. A right join also yields twelve rows here. `validate="many_to_one"` requires unique right-hand keys and stops the invalid join; it does not decide which duplicate to retain. The intentional exception is caught so Run All continues.

In [12]:
airlines_duplicate = pd.concat([airlines, airlines.iloc[[0]]], ignore_index=True)
display(airlines_duplicate)
for how in ['left','right']:
    duplicate_result = flights.merge(airlines_duplicate,left_on='airline_code',right_on='code',how=how)
    display(duplicate_result)
    assert len(duplicate_result) == 12
try:
    flights.merge(airlines_duplicate,left_on='airline_code',right_on='code',how='left',validate='many_to_one')
except pd.errors.MergeError as error:
    print('Expected validation failure:', error)
checked = flights.merge(airlines,left_on='airline_code',right_on='code',how='left',validate='many_to_one')
assert len(checked) == len(flights)
display(checked)

  code        airline_name
0   UA     United Airlines
1   DL     Delta Air Lines
2   WN  Southwest Airlines
3   AA   American Airlines
4   AS     Alaska Airlines
5   UA     United Airlines

   flight_date flight_number origin  ... airline_code  code        airline_name
0   2024-01-15        UA1247    BWI  ...           UA    UA     United Airlines
1   2024-01-15        UA1247    BWI  ...           UA    UA     United Airlines
2   2024-01-22         DL456    ORD  ...           DL    DL     Delta Air Lines
3   2024-02-08        WN2891    LAX  ...           WN    WN  Southwest Airlines
4   2024-02-10        WN1055    PHX  ...           WN    WN  Southwest Airlines
5   2024-03-05         AA892    DEN  ...           AA    AA   American Airlines
6   2024-03-12         UA634    DFW  ...           UA    UA     United Airlines
7   2024-03-12         UA634    DFW  ...           UA    UA     United Airlines
8   2024-04-20        B61840    IAD  ...           B6   NaN                 NaN
9   2024-0

## Column conflicts and chained joins

The toy airport table uses `distance` for miles from city center, not flight miles. The values 25 and 18 are illustrative, not verified geographic facts. When non-key column names overlap, pandas adds `_x` and `_y` by default. Inspect their origins before interpreting them. The two inner joins retain only matching origins in the toy table.

In [13]:
airport_info = pd.DataFrame({'code':['BWI','ORD'],'city':['Baltimore','Chicago'],'distance':[25,18]})
display(airport_info)
flights_airports = (flights.merge(airlines,left_on='airline_code',right_on='code',how='inner')
                   .merge(airport_info,left_on='origin',right_on='code',how='inner'))
display(flights_airports)
print('distance_x = flight miles; distance_y = illustrative city-center miles')

  code       city  distance
0  BWI  Baltimore        25
1  ORD    Chicago        18

  flight_date flight_number origin  ... code_y       city distance_y
0  2024-01-15        UA1247    BWI  ...    BWI  Baltimore         25
1  2024-01-22         DL456    ORD  ...    ORD    Chicago         18

[2 rows x 15 columns]

distance_x = flight miles; distance_y = illustrative city-center miles


## Averaging averages: what gets equal weight?

An average of the five airline means gives each airline equal weight: 764.5 miles. The mean of all ten flights gives each flight equal weight: 831.1 miles. Delta has three flights while B6 has one. Both calculations execute correctly but answer different questions. This difference alone is not Simpson’s paradox or proof of bias. Recover the flight mean using group totals and counts.

In [14]:
distance_summary = flights.groupby('airline_code').agg(flights=('flight_number','size'), total_miles=('distance','sum'), mean_miles=('distance','mean'))
display(distance_summary)
print('Airline-weighted:', distance_summary['mean_miles'].mean())
print('Flight-weighted:', flights['distance'].mean())
recovered = distance_summary['total_miles'].sum() / distance_summary['flights'].sum()
assert abs(recovered - 831.1) < 1e-9
print('Recovered from totals and counts:', recovered)

              flights  total_miles  mean_miles
airline_code                                  
AA                  2         1758       879.0
B6                  1          429       429.0
DL                  3         3285      1095.0
UA                  2         1867       933.5
WN                  2          972       486.0

Airline-weighted: 764.5
Flight-weighted: 831.1
Recovered from totals and counts: 831.1


## Supplemental practice

1. Summarize flight count, known-price count, and mean recorded price per airline.
2. Preserve all flights while adding airline names and validating the lookup.
3. Identify unmatched flight airline codes.
4. Explain why missing delays cannot automatically be treated as zero.
5. Which weighting answers “average flight distance in this log”?

In [15]:
# Write your practice code here.


## Practice solutions

Use row count for flights and nonmissing count for known prices. Supplemental `indicator=True` marks match provenance; it is an added debugging aid, not a required lecture operation.

In [16]:
summary = flights.groupby('airline_code').agg(flight_count=('flight_number','size'), known_prices=('price','count'), mean_price=('price','mean')).reset_index()
display(summary)
matched = flights.merge(airlines,left_on='airline_code',right_on='code',how='left',validate='many_to_one',indicator=True)
unmatched = matched.loc[matched['_merge'] == 'left_only','airline_code'].unique().tolist()
assert unmatched == ['B6']
print('Unmatched:', unmatched)
print('Missing delays are unknown, not established zero delays.')
print('The average flight uses equal weight per flight: 831.1 miles.')

  airline_code  flight_count  known_prices  mean_price
0           AA             2             2     255.775
1           B6             1             1     179.500
2           DL             3             2     346.000
3           UA             2             2     317.375
4           WN             2             1     149.990

Unmatched: ['B6']
Missing delays are unknown, not established zero delays.
The average flight uses equal weight per flight: 831.1 miles.


## Quick review

| Question | Operation |
|---|---|
| Summarize a table | `describe()` |
| Count missing by field / row | `isna().sum()` / `isna().sum(axis=1)` |
| Count categories | `value_counts()` |
| Count all group rows | `groupby(...).size()` |
| Count recorded group values | `groupby(...)["field"].count()` |
| Several named summaries | `groupby(...).agg(name=("field", "operation"))` |
| Attach lookup fields | `merge(..., left_on=..., right_on=..., how=...)` |
| Enforce unique lookup keys | `validate="many_to_one"` |

Before trusting a result: check row meaning, missingness, keys, row counts, duplicates, units, and weights.